# BIOMED UMSS — Entrenamiento del clasificador de cromosomas (EfficientNet-B3)

Fase C2 (ADR-0007 / DD-ML-001). Entrena el clasificador que reemplaza al
`PlaceholderClassifier` de `backend-ml`, sobre los **19.845 crops etiquetados**
extraídos de los cariogramas MetaClass (Fase C1).

**Cómo usar:**
- **Colab:** Runtime → *Change runtime type* → **GPU (T4)**. Subí `crops.zip`
  cuando la celda lo pida.
- **Kaggle:** New Notebook → *Accelerator* → **GPU P100/T4**. Subí `crops.zip`
  como *Dataset* y ajustá `DATA_DIR`.

Estrategia: **transfer learning** — EfficientNet-B3 pre-entrenado (ImageNet),
se congela el backbone y se entrena la cabeza de 24 clases; luego un fine-tune
corto. Salida: `classifier.pth` + `classes.json` + `model_meta.json` para C3.


In [ ]:
# 1) Setup
import os, json, time, zipfile, random
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
import torchvision
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| device:', device)
if device == 'cpu':
    print('AVISO: sin GPU -> el entrenamiento sera lento. Activa GPU en el runtime.')


In [ ]:
# 2) Config
IMG_SIZE = 224          # EfficientNet-B3 nativo es 300; 224 entrena mas rapido
BATCH = 64
EPOCHS_HEAD = 6         # fase 1: solo la cabeza (backbone congelado)
EPOCHS_FT = 4           # fase 2: fine-tune del backbone (LR bajo)
VAL_FRAC = 0.15
DATA_DIR = 'crops'      # carpeta con subdirs por clase: 1/ 2/ ... 22/ X/ Y/
OUT_DIR = 'model_out'
os.makedirs(OUT_DIR, exist_ok=True)


## Subir el dataset
En **Colab**, corré la celda siguiente y subí `crops.zip` (se descomprime a `crops/`).
En **Kaggle**, subí `crops.zip` como Dataset y poné `DATA_DIR = '/kaggle/input/<tu-dataset>/crops'` arriba.

In [ ]:
# 3) (Colab) subir + descomprimir crops.zip
if not os.path.isdir(DATA_DIR):
    try:
        from google.colab import files
        print('Subi crops.zip ...')
        up = files.upload()
        zname = next(iter(up))
        with zipfile.ZipFile(zname) as z: z.extractall(DATA_DIR)
        print('Descomprimido en', DATA_DIR)
    except Exception as e:
        print('No es Colab o fallo la subida:', e)
        print('Asegurate de que DATA_DIR apunte a la carpeta con subdirs por clase.')
assert os.path.isdir(DATA_DIR), f'No existe {DATA_DIR}'
print('clases:', sorted(os.listdir(DATA_DIR)))


In [ ]:
# 4) Datasets, split y balanceo de clases
IMAGENET_MEAN = [0.485, 0.456, 0.406]; IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full = datasets.ImageFolder(DATA_DIR)              # class_to_idx alfabetico
classes = full.classes
print('num clases:', len(classes), classes)
n_val = int(len(full) * VAL_FRAC); n_train = len(full) - n_val
g = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(full, [n_train, n_val], generator=g)

# transforms distintos por split (wrapper)
class TfSubset(torch.utils.data.Dataset):
    def __init__(self, sub, tf): self.sub, self.tf = sub, tf
    def __len__(self): return len(self.sub)
    def __getitem__(self, i):
        img, y = self.sub.dataset.samples[self.sub.indices[i]]
        from PIL import Image
        return self.tf(Image.open(img).convert('RGB')), y
train_data = TfSubset(train_ds, train_tf)
val_data = TfSubset(val_ds, eval_tf)

# balanceo: WeightedRandomSampler por frecuencia de clase (Y esta sub-representado)
targets = [full.samples[i][1] for i in train_ds.indices]
counts = np.bincount(targets, minlength=len(classes))
class_w = 1.0 / np.clip(counts, 1, None)
sample_w = [class_w[t] for t in targets]
sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(train_data, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print('train:', len(train_data), '| val:', len(val_data))
print('conteo por clase (train):', dict(zip(classes, counts.tolist())))


In [ ]:
# 5) Modelo: EfficientNet-B3 pre-entrenado + cabeza de 24 clases
weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1
model = models.efficientnet_b3(weights=weights)
in_feats = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_feats, len(classes))
model = model.to(device)

def set_backbone_trainable(flag):
    for p in model.features.parameters(): p.requires_grad = flag

criterion = nn.CrossEntropyLoss()


In [ ]:
# 6) Bucle de entrenamiento + evaluacion
def evaluate():
    model.eval(); correct = 0; total = 0
    all_y = []; all_p = []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x); pred = out.argmax(1)
            correct += (pred == y).sum().item(); total += y.size(0)
            all_y += y.cpu().tolist(); all_p += pred.cpu().tolist()
    acc = correct / total
    # macro-F1 sin sklearn
    import collections
    f1s = []
    for c in range(len(classes)):
        tp = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp == c)
        fp = sum(1 for yy, pp in zip(all_y, all_p) if yy != c and pp == c)
        fn = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp != c)
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return acc, float(np.mean(f1s)), all_y, all_p

def train_phase(epochs, lr, train_backbone):
    set_backbone_trainable(train_backbone)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    for ep in range(epochs):
        model.train(); t0 = time.time(); running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); out = model(x); loss = criterion(out, y)
            loss.backward(); opt.step(); running += loss.item()
        acc, f1, *_ = evaluate()
        print(f'  ep {ep+1}/{epochs}  loss={running/len(train_loader):.3f}  val_acc={acc:.3f}  val_macroF1={f1:.3f}  ({time.time()-t0:.0f}s)')


In [ ]:
# 7) Fase 1 — cabeza (backbone congelado)
print('== Fase 1: entrenar la cabeza ==')
train_phase(EPOCHS_HEAD, lr=1e-3, train_backbone=False)


In [ ]:
# 8) Fase 2 — fine-tune del backbone (LR bajo)
print('== Fase 2: fine-tune ==')
train_phase(EPOCHS_FT, lr=1e-4, train_backbone=True)


In [ ]:
# 9) Evaluacion final + reporte por clase
acc, f1, all_y, all_p = evaluate()
print(f'ACCURACY={acc:.4f}  MACRO-F1={f1:.4f}')
print('\nPor clase (precision/recall/soporte):')
import numpy as np
for c in range(len(classes)):
    tp = sum(1 for yy,pp in zip(all_y,all_p) if yy==c and pp==c)
    fp = sum(1 for yy,pp in zip(all_y,all_p) if yy!=c and pp==c)
    fn = sum(1 for yy,pp in zip(all_y,all_p) if yy==c and pp!=c)
    sup = sum(1 for yy in all_y if yy==c)
    prec = tp/(tp+fp) if tp+fp else 0; rec = tp/(tp+fn) if tp+fn else 0
    print(f'  {classes[c]:>2}: P={prec:.2f} R={rec:.2f} n={sup}')


In [ ]:
# 10) Guardar el modelo entrenado (para Fase C3 en backend-ml)
torch.save(model.state_dict(), f'{OUT_DIR}/classifier.pth')
json.dump(classes, open(f'{OUT_DIR}/classes.json','w'))
json.dump({
    'arch': 'efficientnet_b3', 'img_size': IMG_SIZE, 'num_classes': len(classes),
    'normalization': {'mean': IMAGENET_MEAN, 'std': IMAGENET_STD},
    'val_accuracy': round(acc,4), 'val_macro_f1': round(f1,4),
}, open(f'{OUT_DIR}/model_meta.json','w'), indent=2)
print('guardado en', OUT_DIR, os.listdir(OUT_DIR))


In [ ]:
# 11) Descargar (Colab)
try:
    from google.colab import files
    files.download(f'{OUT_DIR}/classifier.pth')
    files.download(f'{OUT_DIR}/classes.json')
    files.download(f'{OUT_DIR}/model_meta.json')
except Exception as e:
    print('En Kaggle: descarga classifier.pth / classes.json / model_meta.json desde Output.', e)


## Siguiente paso (Fase C3)
Descargá **`classifier.pth`**, **`classes.json`** y **`model_meta.json`**, y ponelos
en `backend-ml/models/`. El adaptador `EfficientNetClassifier(ClassifierPort)`
los carga y reemplaza al `PlaceholderClassifier` — sin tocar la API ni el pipeline
(diseño hexagonal, ADR-0007).